# Ablation Study — The Instructor

CS 372 Applied ML · Duke University · Spring 2026

---

## Design

Two independent design choices are ablated:

| Choice | Description |
|--------|-------------|
| **Analyst context** | Structured JSON from the silent analyst layer, prepended to the instructor's first user message |
| **RAG examples** | Top-3 structurally similar professional screenplay scenes, injected into the system prompt |

**4 conditions (2×2 factorial):**

| Condition | Analyst | RAG | Hypothesis |
|-----------|---------|-----|------------|
| Baseline | ❌ | ❌ | Generic craft advice, low specificity |
| Analyst only | ✅ | ❌ | Grounded in structure, no professional examples |
| RAG only | ❌ | ✅ | Example-grounded but unaware of this screenplay's specific problems |
| **Full system** | ✅ | ✅ | Best of both: structurally aware + example-grounded |

**Quality metrics (automated):**
- `question_count` — Socratic quality: how many questions does the instructor ask?
- `specificity` — named character or location references from the submitted excerpt
- `craft_terms` — mentions of craft vocabulary (scene value, subtext, gap, want/need, etc.)
- `prescription_score` — inverse: prescriptive language = Socratic failure

**Test set:** 5 scene chunks sampled from the corpus, diverse in genre and act position.

In [ ]:
import json
import re
import sys
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ROOT = Path('../').resolve()
sys.path.insert(0, str(ROOT / 'backend'))

plt.rcParams.update({
    'figure.facecolor': '#1a1714',
    'axes.facecolor':   '#242018',
    'axes.edgecolor':   '#4a4035',
    'axes.labelcolor':  '#d4c5a9',
    'xtick.color':      '#a09070',
    'ytick.color':      '#a09070',
    'text.color':       '#d4c5a9',
    'grid.color':       '#3a3025',
    'grid.alpha':       0.5,
})
COLORS = {
    'baseline':      '#c67a7a',
    'analyst_only':  '#c6a97a',
    'rag_only':      '#7ab8c6',
    'full_system':   '#7ac67a',
}

RESULTS_FILE = ROOT / 'data' / 'ablation_results.json'
print(f"Results file: {RESULTS_FILE}")
print(f"Exists: {RESULTS_FILE.exists()}")

In [ ]:
# Load pre-computed ablation results from scripts/run_ablation.py
# The script ran 4 conditions × 5 excerpts against the live backend (Ollama / qwen2.5:7b-instruct)
# and saved incremental results to data/ablation_results.json

with open(RESULTS_FILE) as f:
    raw_results = json.load(f)

# Normalize: handle both list format (from run_ablation.py) and legacy dict format
if isinstance(raw_results, list):
    results = raw_results
else:
    # Legacy format: reconstruct from nested structure
    results = []
    for cond_data in raw_results.get('conditions', []):
        for i in range(raw_results.get('n_excerpts', 5)):
            results.append({
                'excerpt_id':   f'E{i+1}',
                'condition':    cond_data['name'],
                'use_rag':      cond_data.get('rag', False),
                'needs_analyst': cond_data.get('analyst', False),
                'question_count':    cond_data.get('avg_questions', 0),
                'craft_terms':       cond_data.get('avg_craft', 0),
                'specificity':       cond_data.get('avg_specificity', 0),
                'prescriptions':     cond_data.get('avg_prescriptions', 0),
                'latency_ms':        cond_data.get('avg_latency_ms', 0),
            })

CONDITIONS_ORDER = ['baseline', 'analyst_only', 'rag_only', 'full_system']
CONDITION_LABELS = {
    'baseline':     'Baseline\n(no context)',
    'analyst_only': 'Analyst only\n(no RAG)',
    'rag_only':     'RAG only\n(no analyst)',
    'full_system':  'Full system\n(analyst + RAG)',
}

print(f"Loaded {len(results)} results")
print(f"Conditions found: {sorted(set(r['condition'] for r in results))}")
print(f"Excerpts found:   {sorted(set(r['excerpt_id'] for r in results))}")
print()

# Show raw results
print(f"{'ID':<5} {'Condition':<16} {'Q?':>4} {'Craft':>6} {'Spec':>6} {'Presc':>6} {'ms':>7}")
print("-" * 52)
for r in sorted(results, key=lambda x: (x['excerpt_id'], x['condition'])):
    print(f"{r['excerpt_id']:<5} {r['condition']:<16} {r['question_count']:>4.0f} "
          f"{r['craft_terms']:>6.1f} {r['specificity']:>6.1f} "
          f"{r['prescriptions']:>6.1f} {r.get('latency_ms', 0):>7.0f}")

In [ ]:
# Aggregate by condition
agg = {c: defaultdict(list) for c in CONDITIONS_ORDER}
for r in results:
    c = r['condition']
    if c in agg:
        for metric in ['question_count', 'craft_terms', 'specificity', 'prescriptions', 'latency_ms']:
            agg[c][metric].append(r.get(metric, 0))

summary = {}
for cond in CONDITIONS_ORDER:
    if agg[cond]:
        summary[cond] = {k: round(np.mean(v), 2) for k, v in agg[cond].items()}

print("Mean scores per condition:")
print(f"{'Condition':<16} {'Questions':>9} {'Craft':>7} {'Specific':>9} {'Prescribe':>10} {'Latency':>9}")
print('-' * 64)
for cond in CONDITIONS_ORDER:
    s = summary.get(cond, {})
    if s:
        print(f"{cond:<16} {s['question_count']:>9.1f} {s['craft_terms']:>7.1f} "
              f"{s['specificity']:>9.1f} {s['prescriptions']:>10.1f} {s['latency_ms']:>8.0f}ms")

In [ ]:
# Part 1: RAG Pipeline ablation — cosine vs reranked retrieval (no LLM required)
import requests
try:
    import chromadb
    from rag.retrieve import _get_embedder, _get_collection, retrieve as retrieve_full
    BACKEND_AVAILABLE = True
except ImportError:
    BACKEND_AVAILABLE = False
    print("rag module not importable from this notebook context — skipping Part 1")

# Load test excerpts (same seed as run_ablation.py)
chunks = [json.loads(l) for l in (ROOT / 'data' / 'chunks' / 'chunks.jsonl').read_text().splitlines() if l.strip()]

random.seed(7)
by_act = defaultdict(list)
for c in chunks:
    by_act[c.get('act_position', 'unknown')].append(c)

TEST_EXCERPTS = []
for pos in ['act_1', 'act_2_first_half', 'act_2_second_half', 'act_3']:
    pool = [c for c in by_act[pos] if len(c.get('text', '')) > 300]
    if pool:
        TEST_EXCERPTS.append(random.choice(pool))

remaining = [c for c in chunks if c not in TEST_EXCERPTS and len(c.get('text', '')) > 300]
if remaining:
    TEST_EXCERPTS.append(random.choice(remaining))
TEST_EXCERPTS = TEST_EXCERPTS[:5]

print(f"Test excerpts ({len(TEST_EXCERPTS)}):")
for i, ex in enumerate(TEST_EXCERPTS):
    print(f"  E{i+1}: [{ex['title']}] {ex.get('scene_heading','')[:50]} ({ex.get('act_position','')})")

In [ ]:
# RAG pipeline ablation: cosine-only vs cosine + cross-encoder rerank
def retrieve_cosine_only(query_text, n=3):
    collection = _get_collection()
    if collection is None:
        return []
    embedder = _get_embedder()
    q_emb = embedder.encode([query_text[:2000]], normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=q_emb, n_results=min(n, collection.count()),
        include=['documents', 'metadatas', 'distances'],
    )
    return [
        {'title': results['metadatas'][0][i].get('title',''),
         'scene_heading': results['metadatas'][0][i].get('scene_heading',''),
         'distance': results['distances'][0][i]}
        for i in range(len(results['ids'][0]))
    ]

rag_ablation = []
if BACKEND_AVAILABLE:
    print('Retrieval ablation across 5 test excerpts...\n')
    for ex in TEST_EXCERPTS:
        query = ex.get('text','')[:1000]
        cosine   = retrieve_cosine_only(query, n=3)
        reranked = retrieve_full(query, n_final=3)

        cosine_ids   = {r['title']+r['scene_heading'] for r in cosine}
        reranked_ids = {r['title']+r.get('scene_heading','') for r in reranked}
        overlap = len(cosine_ids & reranked_ids) / max(1, len(cosine_ids))

        cosine_dist    = np.mean([r['distance'] for r in cosine]) if cosine else 0
        reranked_score = np.mean([r.get('rerank_score', 0) for r in reranked]) if reranked else 0

        rag_ablation.append({
            'title': ex['title'][:25],
            'act':   ex.get('act_position',''),
            'cosine_dist':   round(cosine_dist, 4),
            'rerank_score':  round(reranked_score, 3),
            'overlap':       round(overlap, 2),
            'cosine_top1':   cosine[0]['title'][:22] if cosine else '',
            'reranked_top1': reranked[0]['title'][:22] if reranked else '',
        })
        print(f"[{ex['title'][:30]}]")
        print(f"  Cosine top-1:   {rag_ablation[-1]['cosine_top1']} (dist={cosine_dist:.4f})")
        print(f"  Reranked top-1: {rag_ablation[-1]['reranked_top1']} (score={reranked_score:.3f})")
        print(f"  Result overlap: {overlap:.0%}\n")
else:
    print("Skipped (rag module not available in notebook env)")

---
## Part 2 — Instructor Quality Ablation (4 Conditions)

Results loaded from `data/ablation_results.json`, produced by `scripts/run_ablation.py`.

**Run configuration:** `qwen2.5:7b-instruct` via Ollama (local inference, no API key required).
Each condition called the `/chat` backend endpoint; analyst conditions used pre-generated JSON cached in `data/ablation_cache/`.

**Scoring dimensions (automated):**
- `question_count` — proxy for Socratic quality
- `craft_terms` — from a 20-term craft vocabulary list (scene value, subtext, gap, etc.)
- `specificity` — capitalized named entities + numbers (excerpt-specific references)
- `prescriptions` — prescriptive phrase count (lower = better Socratic adherence)

In [ ]:
# Main bar chart: 3 quality metrics × 4 conditions
metrics       = ['question_count', 'craft_terms', 'specificity']
metric_labels = ['Questions asked\n(Socratic quality)', 'Craft terms\n(grounding)', 'Specificity\n(named refs + numbers)']
cond_labels   = [CONDITION_LABELS[c] for c in CONDITIONS_ORDER]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, label in zip(axes, metrics, metric_labels):
    vals   = [summary.get(c, {}).get(metric, 0) for c in CONDITIONS_ORDER]
    colors = [COLORS[c] for c in CONDITIONS_ORDER]
    bars   = ax.bar(cond_labels, vals, color=colors, alpha=0.88, width=0.55)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.02,
                f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')

    ax.set_title(label, fontsize=10)
    ax.set_ylabel('Mean (5 excerpts)')
    ax.grid(axis='y')
    ax.tick_params(axis='x', labelsize=8)
    ax.set_ylim(0, max(vals) * 1.2 if vals else 1)

patches = [mpatches.Patch(color=COLORS[c], label=l.replace('\n', ' ')) for c, l in zip(CONDITIONS_ORDER, cond_labels)]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=8, bbox_to_anchor=(0.5, -0.04))

plt.suptitle('Ablation Study: Effect of Analyst Context and RAG on Instructor Quality',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'fig_ablation_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → data/fig_ablation_quality.png")

In [ ]:
# Per-excerpt breakdown heatmap
excerpt_ids = sorted(set(r['excerpt_id'] for r in results))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax_i, (ax, metric, label) in enumerate(zip(axes, metrics, metric_labels)):
    matrix = np.zeros((len(CONDITIONS_ORDER), len(excerpt_ids)))
    for ri, row in enumerate(results):
        ci = CONDITIONS_ORDER.index(row['condition']) if row['condition'] in CONDITIONS_ORDER else -1
        ei = excerpt_ids.index(row['excerpt_id']) if row['excerpt_id'] in excerpt_ids else -1
        if ci >= 0 and ei >= 0:
            matrix[ci, ei] = row.get(metric, 0)

    im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(excerpt_ids)))
    ax.set_xticklabels(excerpt_ids, fontsize=9)
    ax.set_yticks(range(len(CONDITIONS_ORDER)))
    ax.set_yticklabels([CONDITION_LABELS[c].replace('\n', ' ') for c in CONDITIONS_ORDER], fontsize=8)
    ax.set_title(label.replace('\n', ' '), fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Annotate cells
    for r in range(matrix.shape[0]):
        for c in range(matrix.shape[1]):
            ax.text(c, r, f'{matrix[r, c]:.0f}', ha='center', va='center', fontsize=8,
                    color='black' if matrix[r, c] > matrix.max()*0.6 else 'white')

plt.suptitle('Quality Metrics Heatmap: Condition × Excerpt', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'fig_ablation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → data/fig_ablation_heatmap.png")

In [ ]:
# Latency comparison
fig2, ax_lat = plt.subplots(figsize=(8, 3.5))
cond_labels_short = ['Baseline', 'Analyst only', 'RAG only', 'Full system']
lat_vals = [summary.get(c, {}).get('latency_ms', 0) for c in CONDITIONS_ORDER]
bars = ax_lat.barh(cond_labels_short, lat_vals, color=[COLORS[c] for c in CONDITIONS_ORDER], alpha=0.85, height=0.5)
for bar, val in zip(bars, lat_vals):
    ax_lat.text(val + max(lat_vals)*0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.0f}ms', va='center', fontsize=9)
ax_lat.set_xlabel('Mean response latency (ms)')
ax_lat.set_title('Response Latency by Condition (Ollama / qwen2.5:7b-instruct)')
ax_lat.grid(axis='x')
ax_lat.set_xlim(0, max(lat_vals)*1.2 if lat_vals else 1)
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'fig_ablation_latency.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → data/fig_ablation_latency.png")

In [ ]:
# Auto-generate conclusions
base  = summary.get('baseline', {})
full  = summary.get('full_system', {})
a_only = summary.get('analyst_only', {})
r_only = summary.get('rag_only', {})

if base and full:
    q_gain  = full.get('question_count', 0) - base.get('question_count', 0)
    c_gain  = full.get('craft_terms', 0)    - base.get('craft_terms', 0)
    s_gain  = full.get('specificity', 0)    - base.get('specificity', 0)
    p_delta = full.get('prescriptions', 0)  - base.get('prescriptions', 0)

    analyst_q_delta = a_only.get('question_count', 0) - base.get('question_count', 0)
    rag_q_delta     = r_only.get('question_count', 0) - base.get('question_count', 0)

    print('── Ablation Conclusions ─────────────────────────────────────────')
    print(f'\n1. Full system vs. baseline (both factors enabled):')
    print(f'   Questions:   {base["question_count"]:.1f} → {full["question_count"]:.1f} ({q_gain:+.1f})')
    print(f'   Craft terms: {base["craft_terms"]:.1f} → {full["craft_terms"]:.1f} ({c_gain:+.1f})')
    print(f'   Specificity: {base["specificity"]:.1f} → {full["specificity"]:.1f} ({s_gain:+.1f})')
    print(f'   Prescriptions: {base["prescriptions"]:.1f} → {full["prescriptions"]:.1f} ({p_delta:+.1f})')

    print(f'\n2. Isolated analyst context effect (analyst_only vs baseline):')
    print(f'   Δ questions = {analyst_q_delta:+.1f}')

    print(f'\n3. Isolated RAG effect (rag_only vs baseline):')
    print(f'   Δ questions = {rag_q_delta:+.1f}')

    additive = analyst_q_delta + rag_q_delta
    interaction = q_gain - additive
    print(f'\n4. Interaction effect (does combining add up?):')
    print(f'   Full system gain: {q_gain:+.1f}')
    print(f'   Additive estimate: {additive:+.1f}')
    print(f'   Interaction (residual): {interaction:+.1f}')
    print(f'   → {"Positive synergy between analyst + RAG" if interaction > 0 else "No interaction effect above additive" if abs(interaction) < 0.5 else "Sub-additive: factors overlap in effect"}')
else:
    print("No summary data — run earlier cells first.")

In [ ]:
# Save final summary back to data/ablation_results.json in a clean format
# (preserves raw results + adds aggregated summary)
output = {
    'raw_results': results,
    'summary': summary,
    'n_excerpts': len(excerpt_ids),
    'conditions': CONDITIONS_ORDER,
    'metrics': metrics,
}
(ROOT / 'data' / 'ablation_results.json').write_text(json.dumps(output, indent=2))
print("Updated data/ablation_results.json with summary")

---
## Figures Generated

| Figure | Description |
|--------|-------------|
| `data/fig_ablation_quality.png` | 3-panel bar chart: questions / craft / specificity by condition |
| `data/fig_ablation_heatmap.png` | Per-excerpt heatmap of quality metrics by condition |
| `data/fig_ablation_latency.png` | Response latency comparison |

## Evidence Summary

- **Script:** `scripts/run_ablation.py` — runs 4 conditions × 5 excerpts, saves incremental results
- **Data:** `data/ablation_results.json` — raw results + aggregated summary
- **Cache:** `data/ablation_cache/` — analyst JSONs (one per excerpt, generated by the analyst layer)
- **Model:** `qwen2.5:7b-instruct` via Ollama (local inference)

---
## Summary and Conclusions

Fill in after running: